In [54]:
!pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\Administrator\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [55]:
import pandas as pd 
import numpy as np
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

In [56]:
credits = pd.read_csv("tmdb_5000_credits.csv")
movies = pd.read_csv("tmdb_5000_movies.csv")


In [57]:
credits.head(0)

,movie_id,title,cast,crew


In [58]:
movies.head(0)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count


In [59]:
movies=movies.merge(credits,left_on='title',right_on='title')

In [60]:
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]


In [61]:
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [62]:
def convert (obj):
    L=[]
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L

In [63]:
movies['genres'] = movies['genres'].apply(convert)

In [64]:
movies['keywords'] = movies['keywords'].apply(convert)

In [65]:
movies['cast'] = movies['cast'].apply(convert)

In [66]:
movies['crew'] = movies['crew'].apply(convert)

In [67]:
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weave...","[Stephen E. Rivkin, Rick Carter, Christopher B..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley, ...","[Dariusz Wolski, Gore Verbinski, Jerry Bruckhe..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...","[Daniel Craig, Christoph Waltz, Léa Seydoux, R...","[Thomas Newman, Sam Mendes, Anna Pinnock, John..."
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...","[Christian Bale, Michael Caine, Gary Oldman, A...","[Hans Zimmer, Charles Roven, Christopher Nolan..."
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...","[Taylor Kitsch, Lynn Collins, Samantha Morton,...","[Andrew Stanton, Andrew Stanton, John Lasseter..."


In [68]:
import ast

def convert_cast(obj):
    # Only use literal_eval if the data is a string
    if isinstance(obj, str):
        L = []
        counter = 0
        for i in ast.literal_eval(obj):
            if counter != 3:
                L.append(i['name'])
                counter += 1
            else:
                break
        return L
    # If it's already a list, just return it as is
    return obj

# Apply the safe function
movies['cast'] = movies['cast'].apply(convert_cast)

In [69]:
def fetch_director(text):
    # Only try to eval if it's a non-empty string
    if isinstance(text, str) and text.strip():
        try:
            L = []
            for i in ast.literal_eval(text):
                if i['job'] == 'Director':
                    L.append(i['name'])
            return L
        except (ValueError, SyntaxError):
            return []
    # If it's already a list (from a previous run), just return it
    elif isinstance(text, list):
        return text
    return []

movies['crew'] = movies['crew'].apply(fetch_director)

In [70]:
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weave...","[Stephen E. Rivkin, Rick Carter, Christopher B..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley, ...","[Dariusz Wolski, Gore Verbinski, Jerry Bruckhe..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...","[Daniel Craig, Christoph Waltz, Léa Seydoux, R...","[Thomas Newman, Sam Mendes, Anna Pinnock, John..."
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...","[Christian Bale, Michael Caine, Gary Oldman, A...","[Hans Zimmer, Charles Roven, Christopher Nolan..."
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...","[Taylor Kitsch, Lynn Collins, Samantha Morton,...","[Andrew Stanton, Andrew Stanton, John Lasseter..."


In [ ]:
movies['tags'] = movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']
movies['tags'] = movies['tags'].apply(lambda x: " ".join(x).lower())


movies['overview'] = movies['overview'].fillna('')

movies['tags'] = movies['overview'] + " " + movies['tags']

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['tags'])

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

with open('movie_data.pkl', 'wb') as file:
    pickle.dump((movies[['movie_id', 'title']], cosine_sim), file)

In [72]:
movies['tags']

0       In the 22nd century, a paraplegic Marine is di...
1       Captain Barbossa, long believed to be dead, ha...
2       A cryptic message from Bond’s past sends him o...
3       Following the death of District Attorney Harve...
4       John Carter is a war-weary, former military ca...
                              ...                        
4804    El Mariachi just wants to play his guitar and ...
4805    A newlywed couple's honeymoon is upended by th...
4806    "Signed, Sealed, Delivered" introduces a dedic...
4807    When ambitious New York attorney Sam is sent t...
4808    Ever since the second grade when he first saw ...
Name: tags, Length: 4809, dtype: str

In [73]:
movies=movies[['movie_id','title','overview','tags']]

In [74]:
movies.head()

,movie_id,title,overview,tags
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","In the 22nd century, a paraplegic Marine is di..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","Captain Barbossa, long believed to be dead, ha..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,A cryptic message from Bond’s past sends him o...
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,Following the death of District Attorney Harve...
4,49529,John Carter,"John Carter is a war-weary, former military ca...","John Carter is a war-weary, former military ca..."


In [75]:
movies['tags']=movies['tags'].apply(lambda x: " ".join(x))

In [76]:
movies['tags']=movies['tags'].apply(lambda x: x.lower())

In [77]:
movies['tags']

0       i n   t h e   2 2 n d   c e n t u r y ,   a   ...
1       c a p t a i n   b a r b o s s a ,   l o n g   ...
2       a   c r y p t i c   m e s s a g e   f r o m   ...
3       f o l l o w i n g   t h e   d e a t h   o f   ...
4       j o h n   c a r t e r   i s   a   w a r - w e ...
                              ...                        
4804    e l   m a r i a c h i   j u s t   w a n t s   ...
4805    a   n e w l y w e d   c o u p l e ' s   h o n ...
4806    " s i g n e d ,   s e a l e d ,   d e l i v e ...
4807    w h e n   a m b i t i o u s   n e w   y o r k ...
4808    e v e r   s i n c e   t h e   s e c o n d   g ...
Name: tags, Length: 4809, dtype: str

In [78]:
movies.head()

,movie_id,title,overview,tags
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","i n t h e 2 2 n d c e n t u r y , a ..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","c a p t a i n b a r b o s s a , l o n g ..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,a c r y p t i c m e s s a g e f r o m ...
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,f o l l o w i n g t h e d e a t h o f ...
4,49529,John Carter,"John Carter is a war-weary, former military ca...",j o h n c a r t e r i s a w a r - w e ...


In [79]:
osine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [80]:

def get_recommendations(title, cosine_sim=cosine_sim):
    
    idx = movies[movies['title'] == title].index[0]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:11]
    movie_indices = [i[0] for i in sim_scores]
    return movies['title'].iloc[movie_indices]



In [81]:
print(get_recommendations('The Dark Knight'))

3                         The Dark Knight Rises
119                               Batman Begins
1197                               The Prestige
428                              Batman Returns
1360                                     Batman
299                              Batman Forever
95                                 Interstellar
9            Batman v Superman: Dawn of Justice
28                               Jurassic World
3859    Batman: The Dark Knight Returns, Part 2
Name: title, dtype: str


In [82]:
print(get_recommendations('Spider-Man'))

30                    Spider-Man 2
5                     Spider-Man 3
38        The Amazing Spider-Man 2
20          The Amazing Spider-Man
529               Tears of the Sun
28                  Jurassic World
1134                    15 Minutes
16                    The Avengers
182                        Ant-Man
37      Oz: The Great and Powerful
Name: title, dtype: str


In [83]:
with open('movie_data.pkl', 'wb') as file:
    pickle.dump((movies, cosine_sim), file)